# Listener clustering and fusion

For each cold-start track, rank all eligible users by combining three scores. All scores use training data only, and every vector follows the sorted eligible-user order.

| Score | Source | Definition |
| --- | --- | --- |
| $\mathrm{pool}(u)$ | C's candidate pool | $\sum_{t \in N(s)} J(s,t)\,\log(1+r_{ut})$ over the 50 tag neighbours $N(s)$ with at least two plays |
| $\mathrm{cluster}(u)$ | Listener clustering | Cosine of $u$'s L2-normalised ALS vector with the k-means centres of the top-$N$ pool users |
| $\mathrm{ridge}(u)$ | B's ALS + Ridge | $x_u^\top \hat v_s$ with the Ridge-predicted track vector $\hat v_s$ |

The A path and the full score are

$$
A(u)=\beta\,\tilde{\mathrm{pool}}(u)+(1-\beta)\,\tilde{\mathrm{cluster}}(u), \qquad
\mathrm{score}(u)=\gamma\,A(u)+(1-\gamma)\,\tilde{\mathrm{ridge}}(u),
$$

where the tilde marks per-track min–max or z-score normalisation. Users are ranked by score. Only exactly equal scores are ordered by training plays, so tracks without any signal fall back to the most active users.

## Weighted k-means

The top-$N$ pool users of a track are clustered on their unit ALS vectors $\bar x_u$ with weights $w_u=\mathrm{pool}(u)$. Seeding follows weighted k-means++, and Lloyd iterations minimise

$$
\sum_{u}w_u\,\|\bar x_u-\mu_{c(u)}\|_2^2 .
$$

Each cluster $c$ has the weight share $\pi_c$. Every eligible user then receives either the share-weighted sum $\sum_c \pi_c \cos(\bar x_u,\mu_c)$ (`wsum`) or the closest-centre similarity $\max_c \cos(\bar x_u,\mu_c)$ (`max`). Because `wsum` equals a cosine with one combined direction, it behaves much like $k=1$. The implementation is in `scripts/models/clustering.py`.

## Selection protocol

All choices use the validation split and Recall@100, with NDCG@100 breaking ties.

1. `tune-a` sets $\gamma=1$ and searches $N\in\{500,1000,2000\}$, $k\in\{1,2,3,5,8\}$, both aggregations, and $\beta\in\{0,0.25,0.5,0.75,1\}$.
2. `tune-full` fixes $N$, $k$, and the aggregation, then searches $\beta$, $\gamma\in\{0,0.1,\dots,1\}$, and both normalisations. Each ablation variant takes its best row.
3. `final` applies the frozen variants once to validation and test and scores the exported predictions with the shared evaluator.

In [1]:
from pathlib import Path
import csv
import json
import sys

from IPython.display import Markdown, display

root = Path.cwd()
if not (root / "pyproject.toml").is_file():
    root = root.parent
sys.path.insert(0, str(root))

from scripts.common.evaluate import evaluate

results = root / "reports/fusion/exact"

def read_csv(name):
    with (results / name).open(encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        for key, value in row.items():
            try:
                row[key] = float(value)
            except ValueError:
                pass
    return rows

def table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + " --- |" * len(header)]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    return "\n".join(lines)

tune_a = read_csv("tune_a_grid.csv")
best_a = json.loads((results / "tune_a_best.json").read_text(encoding="utf-8"))
selected = json.loads((results / "selected.json").read_text(encoding="utf-8"))
display(Markdown(
    f"Validation tracks with listeners: **{best_a['tracks']:,}**. "
    f"Tracks without any candidate user: **{best_a['empty_pool_tracks']}**."
))

Validation tracks with listeners: **1,274**. Tracks without any candidate user: **21**.

## Clustering on the candidate pool

The table reports the best validation Recall@100 over $\beta<1$ for each pool size, aggregation, and number of clusters. With $k=1$, `wsum` and `max` coincide.

In [2]:
fmt = lambda value: f"{value:.4f}"
clusters = sorted({int(row["clusters"]) for row in tune_a if row["clusters"] != "-"})
rows = []
for size in sorted({int(row["pool_size"]) for row in tune_a if row["pool_size"] != "-"}):
    for how in ("wsum", "max"):
        cells = []
        for k in clusters:
            values = [row["recall_at_100"] for row in tune_a if row["pool_size"] == size
                      and row["clusters"] == k and row["aggregation"] == how and row["beta"] < 1]
            cells.append(fmt(max(values)) if values else "—")
        rows.append([size, how, *cells])
display(Markdown(table(["N", "Aggregation", *[f"k = {k}" for k in clusters]], rows)))

rows = []
for beta in sorted({row["beta"] for row in tune_a}):
    best = max((row for row in tune_a if row["beta"] == beta), key=lambda row: row["recall_at_100"])
    rows.append([f"{beta:g}", *(fmt(best[m]) for m in ("recall_at_100", "recall_at_500", "recall_at_1000", "ndcg_at_100"))])
display(Markdown("**Effect of β (best over N, k, and aggregation)**. β = 1 is the candidate pool alone, and β = 0 is clustering alone.\n\n"
                 + table(["β", "Recall@100", "Recall@500", "Recall@1000", "NDCG@100"], rows)))
choice = best_a["best_a"]
display(Markdown(f"Selected clustering: N = {int(choice['pool_size'])}, k = {int(choice['clusters'])}, "
                 f"`{choice['aggregation']}`, β = {choice['beta']:g}."))

| N | Aggregation | k = 1 | k = 2 | k = 3 | k = 5 | k = 8 |
| --- | --- | --- | --- | --- | --- | --- |
| 500 | wsum | 0.1595 | 0.1597 | 0.1598 | 0.1597 | 0.1596 |
| 500 | max | — | 0.1598 | 0.1612 | 0.1587 | 0.1603 |
| 1000 | wsum | 0.1589 | 0.1589 | 0.1591 | 0.1589 | 0.1588 |
| 1000 | max | — | 0.1579 | 0.1594 | 0.1611 | 0.1598 |
| 2000 | wsum | 0.1585 | 0.1585 | 0.1584 | 0.1582 | 0.1584 |
| 2000 | max | — | 0.1580 | 0.1577 | 0.1591 | 0.1587 |

**Effect of β (best over N, k, and aggregation)**. β = 1 is the candidate pool alone, and β = 0 is clustering alone.

| β | Recall@100 | Recall@500 | Recall@1000 | NDCG@100 |
| --- | --- | --- | --- | --- |
| 0 | 0.0994 | 0.2574 | 0.3625 | 0.0924 |
| 0.25 | 0.1529 | 0.3147 | 0.4108 | 0.1521 |
| 0.5 | 0.1612 | 0.3395 | 0.4437 | 0.1640 |
| 0.75 | 0.1602 | 0.3415 | 0.4475 | 0.1642 |
| 1 | 0.1560 | 0.3257 | 0.4216 | 0.1614 |

Selected clustering: N = 500, k = 3, `max`, β = 0.5.

## Fusion weights

The final score weights are $w_\mathrm{pool}=\gamma\beta$, $w_\mathrm{cluster}=\gamma(1-\beta)$, and $w_\mathrm{ridge}=1-\gamma$. Rows below *cluster only* are reported for discussion and are not ablation variants.

In [3]:
names = {"full": "Full system", "no_cluster": "Without clustering", "a_only": "A path only",
         "b_only": "B path only", "pool_only": "Candidate pool only", "cluster_only": "Cluster only",
         "cluster_ridge": "Cluster + Ridge"}
rows = []
for key, row in {**selected["variants"], **selected["other_validation_rows"]}.items():
    if key not in names:
        continue
    rows.append([names[key], row["norm"], *(f"{float(row[w]):.3g}" for w in ("w_pool", "w_cluster", "w_ridge")),
                 *(fmt(float(row[m])) for m in ("recall_at_100", "recall_at_1000", "ndcg_at_100"))])
display(Markdown(table(["Variant", "Normalisation", "w_pool", "w_cluster", "w_ridge",
                        "Recall@100", "Recall@1000", "NDCG@100"], rows)))

grid = read_csv("tune_full_grid.csv")
full = sorted((r for r in grid if r["w_pool"] > 0 and r["w_cluster"] > 0 and r["w_ridge"] > 0),
              key=lambda r: -r["recall_at_100"])[:5]
display(Markdown("**Five best full-system rows.** Close values indicate a flat optimum.\n\n" + table(
    ["Normalisation", "β", "γ", "Recall@100", "Recall@1000", "NDCG@100"],
    [[r["norm"], f"{r['beta']:g}", f"{r['gamma']:g}", fmt(r["recall_at_100"]), fmt(r["recall_at_1000"]),
      fmt(r["ndcg_at_100"])] for r in full])))

| Variant | Normalisation | w_pool | w_cluster | w_ridge | Recall@100 | Recall@1000 | NDCG@100 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Full system | zscore | 0.075 | 0.225 | 0.7 | 0.1871 | 0.5047 | 0.1897 |
| Without clustering | minmax | 0.4 | 0 | 0.6 | 0.1837 | 0.5056 | 0.1873 |
| A path only | minmax | 0.5 | 0.5 | 0 | 0.1612 | 0.4437 | 0.1640 |
| B path only | - | 0 | 0 | 1 | 0.1568 | 0.4758 | 0.1522 |
| Candidate pool only | - | 1 | 0 | 0 | 0.1560 | 0.4216 | 0.1614 |
| Cluster only | - | 0 | 1 | 0 | 0.0908 | 0.3647 | 0.0889 |
| Cluster + Ridge | zscore | 0 | 0.4 | 0.6 | 0.1672 | 0.4857 | 0.1603 |

**Five best full-system rows.** Close values indicate a flat optimum.

| Normalisation | β | γ | Recall@100 | Recall@1000 | NDCG@100 |
| --- | --- | --- | --- | --- | --- |
| zscore | 0.25 | 0.3 | 0.1871 | 0.5047 | 0.1897 |
| minmax | 0.75 | 0.4 | 0.1871 | 0.5045 | 0.1887 |
| zscore | 0.25 | 0.4 | 0.1869 | 0.5072 | 0.1887 |
| minmax | 0.75 | 0.5 | 0.1867 | 0.5063 | 0.1878 |
| minmax | 0.5 | 0.6 | 0.1864 | 0.4980 | 0.1864 |

## Ablation results

The shared evaluator scores each exported prediction file. A check mark shows a component with positive weight.

In [4]:
metrics = ["recall_at_100", "recall_at_500", "recall_at_1000", "ndcg_at_100", "ndcg_at_1000"]
labels = ["Recall@100", "Recall@500", "Recall@1000", "NDCG@100", "NDCG@1000"]
finals = {}
for split in ("validation", "test"):
    path = results / f"final_{split}.csv"
    if not path.is_file():
        display(Markdown(f"_No {split} results yet._"))
        continue
    finals[split] = {row["variant"]: row for row in read_csv(path.name)}
    rows = []
    for key, row in finals[split].items():
        weights = selected["variants"][key]
        marks = ["✓" if float(weights[w]) > 0 else "" for w in ("w_pool", "w_cluster", "w_ridge")]
        rows.append([names[key], *marks, *(fmt(row["official_" + m]) for m in metrics)])
    display(Markdown(f"**{split.title()}**\n\n" + table(["Variant", "Pool", "Clusters", "Ridge", *labels], rows)))

rows = []
for split, rows_by_variant in finals.items():
    for name, (a, b) in {"Clustering in the A path (A path only − pool only)": ("a_only", "pool_only"),
                         "Clustering in the full system (full − without clustering)": ("full", "no_cluster"),
                         "Adding Ridge to the A path (full − A path only)": ("full", "a_only")}.items():
        rows.append([split, name, *(f"{rows_by_variant[a]['official_' + m] - rows_by_variant[b]['official_' + m]:+.4f}"
                                    for m in ("recall_at_100", "recall_at_1000", "ndcg_at_100"))])
display(Markdown("**Differences between variants**\n\n" + table(
    ["Split", "Comparison", "ΔRecall@100", "ΔRecall@1000", "ΔNDCG@100"], rows)))

**Validation**

| Variant | Pool | Clusters | Ridge | Recall@100 | Recall@500 | Recall@1000 | NDCG@100 | NDCG@1000 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Full system | ✓ | ✓ | ✓ | 0.1871 | 0.3931 | 0.5047 | 0.1897 | 0.3031 |
| Without clustering | ✓ |  | ✓ | 0.1837 | 0.3929 | 0.5056 | 0.1873 | 0.3017 |
| A path only | ✓ | ✓ |  | 0.1612 | 0.3395 | 0.4437 | 0.1640 | 0.2633 |
| B path only |  |  | ✓ | 0.1568 | 0.3623 | 0.4758 | 0.1522 | 0.2697 |
| Candidate pool only | ✓ |  |  | 0.1560 | 0.3257 | 0.4216 | 0.1614 | 0.2554 |

**Test**

| Variant | Pool | Clusters | Ridge | Recall@100 | Recall@500 | Recall@1000 | NDCG@100 | NDCG@1000 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Full system | ✓ | ✓ | ✓ | 0.1715 | 0.3668 | 0.4772 | 0.1735 | 0.2856 |
| Without clustering | ✓ |  | ✓ | 0.1709 | 0.3651 | 0.4754 | 0.1714 | 0.2839 |
| A path only | ✓ | ✓ |  | 0.1486 | 0.3135 | 0.4137 | 0.1488 | 0.2451 |
| B path only |  |  | ✓ | 0.1394 | 0.3334 | 0.4497 | 0.1383 | 0.2549 |
| Candidate pool only | ✓ |  |  | 0.1453 | 0.3039 | 0.3950 | 0.1474 | 0.2372 |

**Differences between variants**

| Split | Comparison | ΔRecall@100 | ΔRecall@1000 | ΔNDCG@100 |
| --- | --- | --- | --- | --- |
| validation | Clustering in the A path (A path only − pool only) | +0.0052 | +0.0221 | +0.0026 |
| validation | Clustering in the full system (full − without clustering) | +0.0034 | -0.0008 | +0.0024 |
| validation | Adding Ridge to the A path (full − A path only) | +0.0259 | +0.0611 | +0.0257 |
| test | Clustering in the A path (A path only − pool only) | +0.0033 | +0.0187 | +0.0013 |
| test | Clustering in the full system (full − without clustering) | +0.0006 | +0.0018 | +0.0021 |
| test | Adding Ridge to the A path (full − A path only) | +0.0229 | +0.0635 | +0.0247 |

## B path and the fallback rule

`B path only` ranks by the same Ridge scores as `scripts.models.ridge`. Tracks whose tags are all outside the training vocabulary get a zero Ridge vector, so every user ties. Ridge breaks such ties by user ID, while this pipeline falls back to the most active users. That difference alone separates the two rows below.

In [5]:
ridge_config = json.loads((root / "data/models/ridge_config.json").read_text(encoding="utf-8"))
rows = []
for split in finals:
    _, _, official = evaluate(root / f"data/predictions/ridge_{split}.parquet", split)
    ridge = {f"recall_at_{k}": r for k, r, _ in official} | {f"ndcg_at_{k}": n for k, _, n in official}
    zero = ridge_config["evaluation"][split]["zero_feature_tracks"]
    rows.append([split, zero, "Ridge (user-ID ties)", *(fmt(ridge[m]) for m in metrics)])
    rows.append([split, zero, "B path only (activity ties)", *(fmt(finals[split]["b_only"]["official_" + m]) for m in metrics)])
display(Markdown(table(["Split", "Zero-feature tracks", "Ranking", *labels], rows)))

| Split | Zero-feature tracks | Ranking | Recall@100 | Recall@500 | Recall@1000 | NDCG@100 | NDCG@1000 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| validation | 9 | Ridge (user-ID ties) | 0.1568 | 0.3619 | 0.4752 | 0.1522 | 0.2694 |
| validation | 9 | B path only (activity ties) | 0.1568 | 0.3623 | 0.4758 | 0.1522 | 0.2697 |
| test | 11 | Ridge (user-ID ties) | 0.1394 | 0.3329 | 0.4491 | 0.1382 | 0.2547 |
| test | 11 | B path only (activity ties) | 0.1394 | 0.3334 | 0.4497 | 0.1383 | 0.2549 |